In [0]:
-- Check available water indicator rows and breakdown labels
SELECT
  year_id,
  region_id,
  indicator_id,
  breakdown_type,
  breakdown_value,
  measurement
FROM pondolens.dev_gold.fact_regional_indicator
ORDER BY year_id, region_id
LIMIT 100;

-- Check available regional expenditure rows (bronze source; gold table not yet created)
SELECT
  CAST(e.fiscal_year AS INT) AS year_id,
  r.region_id,
  e.agency AS particular_id,
  CAST(e.amount_obligated AS DOUBLE) AS obligation_amount
FROM pondolens.dev_bronze.regional_expenditure_obligations e
JOIN pondolens.dev_silver.shared_region_crosswalk x ON x.source_region = e.region
JOIN pondolens.dev_gold.dim_region r ON r.region_code = x.region_code
WHERE CAST(e.fiscal_year AS INT) = 2024
LIMIT 100;


-- VALIDATION 1: Overview — row count, distinct dimensions

SELECT
  'fact_regional_indicator' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT year_id) AS distinct_years,
  COUNT(DISTINCT region_id) AS distinct_regions,
  COUNT(DISTINCT indicator_id) AS distinct_indicators,
  COUNT(DISTINCT breakdown_type) AS distinct_breakdown_types,
  COUNT(DISTINCT breakdown_value) AS distinct_breakdown_values,
  MIN(year_id) AS min_year_id,
  MAX(year_id) AS max_year_id,
  MIN(measurement) AS min_measurement,
  MAX(measurement) AS max_measurement
FROM pondolens.dev_gold.fact_regional_indicator;


-- VALIDATION 2: Null checks on key columns

SELECT
  SUM(CASE WHEN fact_indicator_id IS NULL THEN 1 ELSE 0 END) AS null_fact_indicator_id,
  SUM(CASE WHEN year_id IS NULL THEN 1 ELSE 0 END) AS null_year_id,
  SUM(CASE WHEN indicator_id IS NULL THEN 1 ELSE 0 END) AS null_indicator_id,
  SUM(CASE WHEN region_id IS NULL THEN 1 ELSE 0 END) AS null_region_id,
  SUM(CASE WHEN measurement IS NULL THEN 1 ELSE 0 END) AS null_measurement,
  SUM(CASE WHEN indicator IS NULL THEN 1 ELSE 0 END) AS null_indicator_desc,
  SUM(CASE WHEN breakdown_type IS NULL THEN 1 ELSE 0 END) AS null_breakdown_type,
  SUM(CASE WHEN breakdown_value IS NULL THEN 1 ELSE 0 END) AS null_breakdown_value,
  SUM(CASE WHEN unit IS NULL THEN 1 ELSE 0 END) AS null_unit,
  SUM(CASE WHEN service IS NULL THEN 1 ELSE 0 END) AS null_service
FROM pondolens.dev_gold.fact_regional_indicator;


-- VALIDATION 3: Referential integrity — orphaned FKs

SELECT
  'orphaned_year_id' AS check_name,
  COUNT(*) AS orphaned_rows
FROM pondolens.dev_gold.fact_regional_indicator f
LEFT JOIN pondolens.dev_gold.dim_year y ON f.year_id = y.year_id
WHERE y.year_id IS NULL

UNION ALL

SELECT
  'orphaned_indicator_id' AS check_name,
  COUNT(*) AS orphaned_rows
FROM pondolens.dev_gold.fact_regional_indicator f
LEFT JOIN pondolens.dev_gold.dim_indicator i ON f.indicator_id = i.indicator_id
WHERE i.indicator_id IS NULL

UNION ALL

SELECT
  'orphaned_region_id' AS check_name,
  COUNT(*) AS orphaned_rows
FROM pondolens.dev_gold.fact_regional_indicator f
LEFT JOIN pondolens.dev_gold.dim_region r ON f.region_id = r.region_id
WHERE r.region_id IS NULL;


-- VALIDATION 4: Duplicate detection on natural key

SELECT
  year_id,
  indicator_id,
  region_id,
  breakdown_type,
  breakdown_value,
  COUNT(*) AS duplicate_count
FROM pondolens.dev_gold.fact_regional_indicator
GROUP BY year_id, indicator_id, region_id, breakdown_type, breakdown_value
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;


-- VALIDATION 5: Measurement range (percentage indicators 0-100)

SELECT
  'measurement_below_0' AS check_name,
  COUNT(*) AS row_count
FROM pondolens.dev_gold.fact_regional_indicator
WHERE measurement < 0

UNION ALL

SELECT
  'measurement_above_100' AS check_name,
  COUNT(*) AS row_count
FROM pondolens.dev_gold.fact_regional_indicator
WHERE measurement > 100;

-- VALIDATION 6: Denormalization consistency (fact vs dim_indicator)

SELECT
  'indicator_mismatch' AS check_name,
  COUNT(*) AS mismatch_count
FROM pondolens.dev_gold.fact_regional_indicator f
JOIN pondolens.dev_gold.dim_indicator i ON f.indicator_id = i.indicator_id
WHERE COALESCE(f.indicator, '') != COALESCE(i.indicator, '')
   OR COALESCE(f.unit, '') != COALESCE(i.unit, '')
   OR COALESCE(f.service, '') != COALESCE(i.service, '')
   OR COALESCE(f.breakdown_type, '') != COALESCE(i.breakdown_type, '')
   OR COALESCE(f.breakdown_value, '') != COALESCE(i.breakdown_value, '')
   OR COALESCE(f.price_basis, '') != COALESCE(i.price_basis, '');


-- VALIDATION 7: Silver-to-Gold row count reconciliation

SELECT
  'silver_source' AS layer,
  COUNT(*) AS row_count
FROM pondolens.dev_silver.service_access

UNION ALL

SELECT
  'gold_fact' AS layer,
  COUNT(*) AS row_count
FROM pondolens.dev_gold.fact_regional_indicator;


-- VALIDATION 8: Coverage matrix — year x indicator

SELECT
  year_id,
  indicator_id,
  indicator,
  COUNT(DISTINCT region_id) AS region_count,
  COUNT(*) AS row_count
FROM pondolens.dev_gold.fact_regional_indicator
GROUP BY year_id, indicator_id, indicator
ORDER BY year_id, indicator_id;

--check for indicators with multiple regional rows:
SELECT
  year_id,
  indicator_id,
  indicator,
  breakdown_type,
  breakdown_value,
  COUNT(DISTINCT region_id) AS region_count,
  COUNT(*) AS row_count
FROM pondolens.dev_gold.fact_regional_indicator
GROUP BY year_id, indicator_id, indicator, breakdown_type, breakdown_value
HAVING COUNT(DISTINCT region_id) > 1
ORDER BY year_id, indicator_id, breakdown_type, breakdown_value;

--Check 2024 obligations sa source:
SELECT
  CAST(fiscal_year AS INT) AS year_id,
  COUNT(*) AS row_count,
  COUNT(DISTINCT region) AS region_count
FROM pondolens.dev_bronze.regional_expenditure_obligations
WHERE CAST(fiscal_year AS INT) = 2024
GROUP BY CAST(fiscal_year AS INT);

--available fiscal years
SELECT
  fiscal_year,
  COUNT(*) AS row_count,
  COUNT(DISTINCT region) AS region_count
FROM pondolens.dev_bronze.regional_expenditure_obligations
GROUP BY fiscal_year
ORDER BY fiscal_year;

-- 2022 obligations vs. basic drinking-water access

WITH regional_budget AS (
  SELECT
    CAST(e.fiscal_year AS INT) AS year_id,
    r.region_id,

    -- Raw total, in pesos
    SUM(TRY_CAST(e.amount_obligated AS DOUBLE)) AS obligations_pesos,

    -- Converted total, in millions of pesos
    SUM(TRY_CAST(e.amount_obligated AS DOUBLE)) / 1000000.0
      AS obligations_millions

  FROM pondolens.dev_bronze.regional_expenditure_obligations e

  JOIN pondolens.dev_silver.shared_region_crosswalk x
    ON x.source_region = e.region

  JOIN pondolens.dev_gold.dim_region r
    ON r.region_code = x.region_code

  WHERE CAST(e.fiscal_year AS INT) = 2022
    AND TRY_CAST(e.amount_obligated AS DOUBLE) IS NOT NULL

  GROUP BY
    CAST(e.fiscal_year AS INT),
    r.region_id
),

water_access AS (
  SELECT
    year_id,
    region_id,
    measurement AS water_access_measurement
  FROM pondolens.dev_gold.fact_regional_indicator
  WHERE year_id = 2022
    AND indicator_id = 1
    AND breakdown_type = 'Total'
    AND breakdown_value = 'All families'
)

SELECT
  b.year_id,
  r.region_code AS region_name,
  b.obligations_pesos,
  b.obligations_millions,
  w.water_access_measurement
FROM regional_budget b

JOIN water_access w
  ON b.year_id = w.year_id
 AND b.region_id = w.region_id

JOIN pondolens.dev_gold.dim_region r
  ON r.region_id = b.region_id

ORDER BY region_name;


--Description:
--- This scatter plot compares each region’s 2022 government obligations with its basic drinking-water access.
--Interpretation:
--- It shows whether spending and water access appear related. It does not prove that spending caused the access level.


Databricks visualization. Run in Databricks to view.